# OS3 — Génération de contenu adaptatif (LLM)

**Objectif :** générer du contenu pédagogique personnalisé selon l'état émotionnel (Groq / Llama 3.1).

**Prérequis :**
- fichier `.env` à la racine du projet : `GROQ_API_KEY=votre_cle`
- `data/scenarios_avec_strategies.csv` (OS2)

**Ordre d'exécution :** Connexion Groq → 20 templates → tests → corpus (optionnel)

**Limite Groq (gratuit) :** ~6000 tokens/minute. En cas d'erreur 429, attendez 10 s ou relancez la cellule. La génération du corpus 200+ est longue — exécutez-la seule en dernier.

In [1]:
from groq import Groq
import time
from dotenv import load_dotenv
import os

load_dotenv()
client = Groq(api_key=os.getenv('GROQ_API_KEY'))

def groq_create_with_retry(**kwargs):
    """Relance automatiquement si limite Groq (429 TPM)."""
    for attempt in range(6):
        try:
            return client.chat.completions.create(**kwargs)
        except Exception as exc:
            msg = str(exc).lower()
            if "429" in msg or "rate_limit" in msg:
                wait = 10 + attempt * 5
                print(f"⏳ Limite Groq atteinte — pause {wait}s puis nouvel essai...")
                time.sleep(wait)
            else:
                raise
    raise RuntimeError("Limite Groq toujours atteinte après plusieurs essais.")

response = groq_create_with_retry(
    model="llama-3.1-8b-instant",
    messages=[
        {"role": "user", "content": "Dis bonjour en français en une phrase."}
    ],
    max_tokens=50
)

print("=== TEST CONNEXION GROQ ===\n")
print(response.choices[0].message.content)
print("\n✅ Connexion Groq réussie !")

=== TEST CONNEXION GROQ ===

Bonjour, comment allez-vous aujourd'hui ?

✅ Connexion Groq réussie !


In [2]:
# ============================================
# OS3 — 20 TEMPLATES 
# ============================================

def generer_contenu_complet(template_id, concept, niveau, duree, module="Informatique"):
    
    templates = {
        # ========== TYPE A — EXPLICATION ALTERNATIVE ==========
        "A1": f"""L'apprenant montre une confusion de niveau {niveau} sur le concept de {concept} depuis {duree} clips (20 secondes). Génère une explication alternative en 200 mots qui :
1. Reformule le concept avec des mots plus simples
2. Utilise une analogie concrète de la vie quotidienne
3. Décompose en 3 étapes claires
4. Termine par une question de vérification de compréhension
Ton encourageant et rassurant.""",

        "A2": f"""L'apprenant est en confusion intense (niveau {niveau}) sur {concept} depuis {duree} clips. Génère une micro-explication de 150 mots qui :
1. Part d'un exemple concret très simple
2. Construit progressivement vers le concept abstrait
3. Utilise des métaphores visuelles
4. Propose un schéma textuel simple (ASCII art acceptable)
Ton très simple, patient et décomposé.""",

        "A3": f"""L'apprenant combine confusion (niveau {niveau}) et frustration (niveau {niveau}) sur {concept}. Génère une aide de déblocage en 200 mots qui :
1. Commence par une phrase rassurante ("Ce concept est complexe, c'est normal")
2. Identifie LE point de blocage probable
3. Propose une approche radicalement différente (visuelle/narrative)
4. Termine par un mini-exercice guidé très simple
Ton empathique et déculpabilisant.""",

        "A4": f"""L'apprenant a montré plusieurs épisodes de confusion sur des concepts similaires à {concept}. Génère une explication de fond en 250 mots qui :
1. Adresse le concept fondamental sous-jacent
2. Crée des ponts avec ce qui a déjà été compris
3. Propose une représentation mentale unificatrice
4. Suggère une stratégie d'apprentissage pour ce type de concept
Ton constructif et orienté stratégie.""",

        "A5": f"""L'apprenant est confus (niveau {niveau}) sur un concept particulièrement abstrait : {concept}. Génère une explication en 200 mots qui :
1. Utilise une analogie physique/tangible
2. Crée un mini-scénario narratif illustratif
3. Alterne entre concret et abstrait graduellement
4. Termine par une reformulation simple du concept
Ton pédagogue et créatif.""",

        # ========== TYPE B — EXEMPLE CONCRET ==========
        "B1": f"""L'apprenant montre de l'ennui (niveau {niveau}) sur le module {module}. Génère un exemple d'application concrète en 200 mots qui :
1. Part d'une situation réelle et actuelle
2. Montre comment {concept} résout un problème concret
3. Inclut des données/chiffres réels si possible
4. Termine par "Et dans votre quotidien, où pourriez-vous utiliser cela ?"
Ton dynamique et engageant.""",

        "B2": f"""L'apprenant a un engagement très faible (niveau 0-1) depuis {duree} clips. Génère un cas pratique captivant en 250 mots qui :
1. Commence par une question intrigante ou un fait surprenant
2. Présente un cas d'usage inattendu de {concept}
3. Montre l'impact réel/les enjeux
4. Propose une mini-investigation ("Essayez de prédire...")
Ton stimulant et provocateur de curiosité.""",

        "B3": f"""L'apprenant s'ennuie (niveau {niveau}) après une section très théorique sur {concept}. Génère un exemple pratique de mise en œuvre en 200 mots qui :
1. "Passons à la pratique !" - transition claire
2. Décrit un mini-projet concret étape par étape
3. Montre le résultat attendu
4. Suggère une variation à explorer
Ton pratique et orienté action.""",

        "B4": f"""L'apprenant a besoin de voir la pertinence de {concept}. Génère un exemple d'application réelle en 250 mots qui :
1. Choisit un domaine d'application actuel (IA, santé, environnement...)
2. Explique comment {concept} est crucial dans ce contexte
3. Cite une entreprise/projet réel qui l'utilise
4. Montre les compétences professionnelles associées
Ton motivant et orienté carrière.""",

        "B5": f"""L'apprenant pourrait bénéficier d'un exemple interdisciplinaire sur {concept}. Génère un exemple en 200 mots qui :
1. Connecte {concept} à un domaine différent (art, sport, cuisine, musique...)
2. Montre la transversalité du concept
3. Utilise une comparaison inattendue
4. Ouvre sur d'autres applications possibles
Ton créatif et décloisonnant.""",

        # ========== TYPE C — EXERCICE PROGRESSIF ==========
        "C1": f"""L'apprenant s'ennuie (niveau {niveau}) car le contenu semble trop simple sur {concept}. Génère un exercice de niveau avancé en 250 mots qui :
1. Pose un problème complexe nécessitant {concept}
2. Ajoute une contrainte supplémentaire (optimisation, edge case...)
3. Demande de justifier la solution
4. Propose une extension "bonus" encore plus difficile
Ton challengeant et respectueux de leur niveau.""",

        "C2": f"""L'apprenant est bien engagé mais pourrait approfondir {concept}. Génère un exercice de consolidation en 200 mots qui :
1. Combine {concept} avec un concept vu précédemment
2. Présente un problème en 3 étapes de difficulté croissante
3. Fournit des indices progressifs
4. Propose une auto-évaluation
Ton encourageant et progressif.""",

        "C3": f"""L'apprenant a bien compris {concept} mais manque de stimulation. Génère un exercice créatif en 200 mots qui :
1. "Imaginez que vous devez..." (mise en situation ouverte)
2. Demande d'appliquer {concept} à un contexte original
3. Valorise plusieurs solutions possibles
4. Encourage la créativité dans l'approche
Ton ouvert et valorisant de la créativité.""",

        "C4": f"""L'apprenant pourrait bénéficier d'un défi plus conséquent sur {concept}. Génère un mini-projet en 250 mots qui :
1. Présente un objectif concret à atteindre
2. Liste les étapes de réalisation (4-5 étapes)
3. Mentionne les concepts à mobiliser
4. Décrit le livrable attendu
Ton ambitieux et structurant.""",

        "C5": f"""L'apprenant est prêt pour un exercice avancé sur {concept}. Génère un exercice de comparaison en 200 mots qui :
1. Présente 2-3 approches différentes pour résoudre un problème
2. Demande d'identifier forces/faiblesses de chacune
3. Demande de justifier un choix selon le contexte
4. Encourage la pensée critique
Ton analytique et stimulant intellectuellement.""",

        # ========== TYPE D — ENCOURAGEMENT ==========
        "D1": f"""L'apprenant montre de la frustration (niveau {niveau}) sur {concept}. Génère un message d'encouragement en 150 mots qui :
1. Valide la difficulté ("Ce concept challenge beaucoup d'apprenants")
2. Rappelle les progrès déjà réalisés dans le cours
3. Propose une perspective ("Dans 2 modules, vous verrez pourquoi...")
4. Termine par une phrase motivante
Ton empathique et encourageant.""",

        "D2": f"""L'apprenant est très frustré (niveau {niveau}) sur {concept}. Génère un message de soutien en 180 mots qui :
1. "Prenez une pause si nécessaire" - déculpabilisation
2. Normalise la difficulté avec données ("85% trouvent ce passage difficile")
3. Propose une stratégie concrète de déblocage
4. Rappelle que la persévérance est plus importante que la vitesse
Ton bienveillant et déstressant.""",

        "D3": f"""L'apprenant semble découragé (engagement faible depuis {duree} clips) sur {concept}. Génère un message remotivant en 200 mots qui :
1. Pose la question "Qu'est-ce qui vous a amené à ce cours initialement ?"
2. Reconnecte avec les objectifs personnels
3. Montre le chemin parcouru (rappel des acquis)
4. Propose de se fixer un micro-objectif immédiat
Ton empowering et re-centrant.""",

        "D4": f"""L'apprenant est bien engagé (niveau {niveau}) sur {concept}. Génère un renforcement positif en 150 mots qui :
1. Félicite l'engagement et la persévérance
2. Souligne un acquis spécifique récent
3. Anticipe positivement la suite
4. Encourage à maintenir cette dynamique
Ton positif et valorisant.""",

        "D5": f"""L'apprenant a surmonté un passage difficile sur {concept}. Génère un message de félicitation en 150 mots qui :
1. Célèbre la réussite ("Vous avez réussi à...")
2. Explicite la compétence développée
3. Fait le lien avec les prochaines étapes
4. Renforce la confiance en soi
Ton fier et confiant."""
    }

    prompt = templates.get(template_id, templates["A1"])
    
    response = groq_create_with_retry(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role": "system",
                "content": "Tu es un expert en pédagogie et ingénierie de formation. Tu génères du contenu pédagogique adaptatif de haute qualité en français."
            },
            {"role": "user", "content": prompt}
        ],
        max_tokens=600,
        temperature=0.7
    )
    
    return response.choices[0].message.content

print("✅ 20 templates chargés avec succès !")
print("Templates disponibles : A1-A5, B1-B5, C1-C5, D1-D5")

# Alias pour compatibilité avec les anciennes cellules
generer_contenu = generer_contenu_complet

✅ 20 templates chargés avec succès !
Templates disponibles : A1-A5, B1-B5, C1-C5, D1-D5


In [3]:
# ============================================
# OS3 — TEST DES 4 TYPES DE TEMPLATES
# Exécuter d'abord la cellule « 20 TEMPLATES » ci-dessus
# ============================================

if "generer_contenu_complet" not in globals():
    raise NameError(
        "Exécutez d'abord la cellule « 20 TEMPLATES », puis relancez cette cellule."
    )

print("=" * 60)
print("TEST DES 4 TYPES DE CONTENU ADAPTATIF")
print("=" * 60)

# Test A1 — Confusion
print("\n📘 TYPE A1 — Explication alternative (Confusion niveau 2)\n")
contenu_a = generer_contenu_complet("A1", "les algorithmes de tri", 2, 2)
print(contenu_a)

print("\n" + "="*60)

# Test B1 — Ennui
print("\n📗 TYPE B1 — Exemple concret (Ennui niveau 2)\n")
contenu_b = generer_contenu_complet("B1", "les algorithmes de tri", 2, 3)
print(contenu_b)

print("\n" + "="*60)

# Test C1 — Ennui élevé
print("\n📙 TYPE C1 — Exercice avancé (Ennui niveau 3)\n")
contenu_c = generer_contenu_complet("C1", "les algorithmes de tri", 3, 4)
print(contenu_c)

print("\n" + "="*60)

# Test D1 — Frustration
print("\n📕 TYPE D1 — Encouragement (Frustration niveau 2)\n")
contenu_d = generer_contenu_complet("D1", "les algorithmes de tri", 2, 3)
print(contenu_d)

TEST DES 4 TYPES DE CONTENU ADAPTATIF

📘 TYPE A1 — Explication alternative (Confusion niveau 2)

Ne t'inquiète pas, nous allons y arriver ensemble !

Imagine que tu as un placard rempli de vêtements et que tu veux les mettre de l'ordre. Tu peux les classer par couleur, par taille ou par type de vêtement. C'est ce que font les algorithmes de tri : ils organisent des données (comme les vêtements) de manière logique et ordonnée.

Voici comment fonctionnent les algorithmes de tri :

**Étape 1 : Comparer les données**
Tu compares chaque vêtement avec les autres pour voir comment ils sont similaires ou différents.

**Étape 2 : Classer les données**
Tu les groupes par catégorie (couleur, taille, etc.) pour créer des sous-groupes organisés.

**Étape 3 : Organiser les données**
Tu as maintenant des sous-groupes de vêtements ordonnés, comme les robes de couleur rouge, les chemises de taille L, etc.

Pouvez-vous me dire comment fonctionne un algorithme de tri ?


📗 TYPE B1 — Exemple concret (Ennu

In [ ]:
# ============================================
# OS3 — GÉNÉRATION CORPUS COMPLET 200+
# ============================================

import pandas as pd
import time

scenarios = pd.read_csv('../data/scenarios_avec_strategies.csv')

# Mapping étendu — plus de templates par pattern
pattern_templates_complet = {
    'stable'           : ['D4', 'D5', 'C2'],
    'ennui_eleve'      : ['C1', 'B1', 'C3', 'C4', 'B4'],
    'frustration'      : ['D1', 'D2', 'A3', 'D3', 'B3'],
    'confusion'        : ['A1', 'A2', 'A5', 'A4', 'A3'],
    'engagement_faible': ['B2', 'B3', 'D3', 'B5', 'C2'],
    'critique'         : ['A3', 'D2', 'B2', 'D1', 'A1']
}

concept = "les algorithmes de tri"
module  = "Algorithmique"

corpus_complet = []
total = 0

print("=== GÉNÉRATION CORPUS COMPLET 200+ ===\n")

for _, row in scenarios.iterrows():
    pattern = row['pattern']
    templates_list = pattern_templates_complet.get(pattern, ['A1'])
    
    for template_id in templates_list:
        print(f"Génération {template_id} — pattern '{pattern}'...")
        
        try:
            contenu = generer_contenu_complet(
                template_id=template_id,
                concept=concept,
                niveau=int(row['niveau']),
                duree=2,
                module=module
            )
            
            corpus_complet.append({
                'clip_id'    : row['ClipID'],
                'pattern'    : pattern,
                'template_id': template_id,
                'emotion'    : row['emotion'],
                'niveau'     : row['niveau'],
                'contenu'    : contenu,
                'nb_mots'    : len(contenu.split())
            })
            total += 1
            time.sleep(3)
            
        except Exception as e:
            print(f"  Erreur : {e}")
            continue

# Sauvegarder
df_corpus_complet = pd.DataFrame(corpus_complet)
df_corpus_complet.to_csv('../data/corpus_200.csv', index=False)

print(f"\n✅ Corpus complet généré !")
print(f"\n=== STATISTIQUES FINALES ===")
print(f"Total contenus     : {len(df_corpus_complet)}")
print(f"Mots moyens        : {df_corpus_complet['nb_mots'].mean():.0f}")
print(f"Templates utilisés : {df_corpus_complet['template_id'].nunique()}/20")
print(f"\nRépartition par pattern :")
print(df_corpus_complet['pattern'].value_counts())
print(f"\nRépartition par template :")
print(df_corpus_complet['template_id'].value_counts())

=== GÉNÉRATION CORPUS COMPLET 200+ ===

Génération D4 — pattern 'stable'...
Génération D5 — pattern 'stable'...
Génération C2 — pattern 'stable'...
Génération D4 — pattern 'stable'...
Génération D5 — pattern 'stable'...
Génération C2 — pattern 'stable'...
Génération D4 — pattern 'stable'...
Génération D5 — pattern 'stable'...
Génération C2 — pattern 'stable'...
Génération D4 — pattern 'stable'...
Génération D5 — pattern 'stable'...
Génération C2 — pattern 'stable'...
Génération D4 — pattern 'stable'...
Génération D5 — pattern 'stable'...
Génération C2 — pattern 'stable'...
Génération D4 — pattern 'stable'...
Génération D5 — pattern 'stable'...
Génération C2 — pattern 'stable'...
Génération D4 — pattern 'stable'...
Génération D5 — pattern 'stable'...
Génération C2 — pattern 'stable'...
Génération D4 — pattern 'stable'...
Génération D5 — pattern 'stable'...
Génération C2 — pattern 'stable'...
Génération D4 — pattern 'stable'...
Génération D5 — pattern 'stable'...
Génération C2 — pattern 

In [ ]:
# ============================================
# OS3 — STRUCTURE GÉNÉRIQUE UNIVERSELLE
# ============================================

def generer_contenu_universel(
    # CONTEXTE PÉDAGOGIQUE
    sujet_cours,
    module_actuel,
    concept,
    niveau_apprenant,
    objectif_pedagogique,
    # ÉTAT D'ENGAGEMENT
    etat_dominant,
    niveau_intensite,
    duree_etat,
    pattern_detecte,
    moment_cours,
    # TÂCHE DE GÉNÉRATION
    type_contenu,
    objectif_adaptation,
    # CONTRAINTES
    longueur="200 mots",
    format_contenu="Paragraphe",
    ton="Encourageant",
    inclure_obligatoirement="",
    ne_pas_repeter=""
):
    
    # Instructions spécifiques par type
    instructions_specifiques = {
        "A": """- Reformuler le concept avec des mots plus simples
- Utiliser une analogie concrète de la vie quotidienne
- Décomposer en étapes claires
- Terminer par une question de vérification""",

        "B": """- Partir d'une situation réelle et actuelle
- Montrer comment le concept résout un problème concret
- Inclure des données/chiffres réels si possible
- Terminer par une question engageante""",

        "C": """- Poser un problème complexe nécessitant le concept
- Ajouter une contrainte supplémentaire
- Demander de justifier la solution
- Proposer une extension bonus""",

        "D": """- Valider la difficulté de manière empathique
- Rappeler les progrès déjà réalisés
- Proposer une perspective positive
- Terminer par une phrase motivante"""
    }

    # Construction du prompt universel
    prompt = f"""
[CONTEXTE PÉDAGOGIQUE]
Sujet du cours        : {sujet_cours}
Module actuel         : {module_actuel}
Concept en cours      : {concept}
Niveau apprenant      : {niveau_apprenant}
Objectif pédagogique  : {objectif_pedagogique}

[ÉTAT D'ENGAGEMENT DE L'APPRENANT]
État dominant         : {etat_dominant}
Niveau d'intensité    : {niveau_intensite}/3
Durée de l'état       : {duree_etat} clips ({duree_etat*10} secondes)
Pattern détecté       : {pattern_detecte}
Moment du cours       : {moment_cours}

[TÂCHE DE GÉNÉRATION]
Type de contenu       : Type {type_contenu}
Objectif adaptation   : {objectif_adaptation}

[CONTRAINTES]
- Longueur            : {longueur}
- Format              : {format_contenu}
- Ton                 : {ton}
- Inclure obligatoirement : {inclure_obligatoirement if inclure_obligatoirement else "Éléments adaptés au type"}
- Niveau de langage   : Adapté au niveau {niveau_apprenant}
- Ne pas répéter      : {ne_pas_repeter if ne_pas_repeter else "Aucune contrainte spécifique"}

[INSTRUCTIONS SPÉCIFIQUES]
{instructions_specifiques.get(type_contenu, instructions_specifiques["A"])}

Génère maintenant le contenu adaptatif en français.
"""

    response = groq_create_with_retry(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role": "system",
                "content": "Tu es un expert en pédagogie et ingénierie de formation. Tu génères du contenu pédagogique adaptatif de haute qualité en français. Tu respectes strictement les contraintes de longueur, format et ton spécifiés."
            },
            {"role": "user", "content": prompt}
        ],
        max_tokens=600,
        temperature=0.7
    )
    
    return response.choices[0].message.content

print("✅ Structure générique universelle chargée !")

In [ ]:
# ============================================
# TEST DE LA STRUCTURE GÉNÉRIQUE UNIVERSELLE
# ============================================

print("=== TEST STRUCTURE UNIVERSELLE ===\n")

# Test 1 — Confusion modérée
print("📘 TEST 1 — Confusion modérée (Type A)\n")
contenu_a = generer_contenu_universel(
    sujet_cours          = "Informatique — Algorithmique",
    module_actuel        = "Module 3 — Structures de données",
    concept              = "les algorithmes de tri",
    niveau_apprenant     = "Intermédiaire",
    objectif_pedagogique = "Comprendre et implémenter un algorithme de tri",
    etat_dominant        = "Confusion",
    niveau_intensite     = 2,
    duree_etat           = 2,
    pattern_detecte      = "confusion croissante",
    moment_cours         = "Milieu du module",
    type_contenu         = "A",
    objectif_adaptation  = "Clarifier",
    longueur             = "200 mots",
    format_contenu       = "Paragraphe avec étapes",
    ton                  = "Encourageant et rassurant"
)
print(contenu_a)

print("\n" + "="*60 + "\n")

# Test 2 — Ennui élevé
print("📙 TEST 2 — Ennui élevé (Type C)\n")
contenu_c = generer_contenu_universel(
    sujet_cours          = "Informatique — Algorithmique",
    module_actuel        = "Module 3 — Structures de données",
    concept              = "les algorithmes de tri",
    niveau_apprenant     = "Avancé",
    objectif_pedagogique = "Maîtriser les algorithmes de tri complexes",
    etat_dominant        = "Boredom",
    niveau_intensite     = 3,
    duree_etat           = 4,
    pattern_detecte      = "ennui prolongé",
    moment_cours         = "Fin du module",
    type_contenu         = "C",
    objectif_adaptation  = "Stimuler",
    longueur             = "250 mots",
    format_contenu       = "Exercice structuré",
    ton                  = "Challengeant et dynamique"
)
print(contenu_c)

print("\n" + "="*60 + "\n")

# Test 3 — Frustration
print("📕 TEST 3 — Frustration (Type D)\n")
contenu_d = generer_contenu_universel(
    sujet_cours          = "Informatique — Algorithmique",
    module_actuel        = "Module 3 — Structures de données",
    concept              = "les algorithmes de tri",
    niveau_apprenant     = "Débutant",
    objectif_pedagogique = "Comprendre le concept de base du tri",
    etat_dominant        = "Frustration",
    niveau_intensite     = 2,
    duree_etat           = 3,
    pattern_detecte      = "frustration croissante",
    moment_cours         = "Début du module",
    type_contenu         = "D",
    objectif_adaptation  = "Encourager",
    longueur             = "150 mots",
    format_contenu       = "Message motivant",
    ton                  = "Empathique et rassurant"
)
print(contenu_d)